# Refinitiv Download — Part 2
Loads the CSVs saved from Part 1 (daily prices, fundamentals, constituents),
then downloads JCI index, BI rate, assembles the master panel, retries failed batches, and runs quality checks.

In [ ]:
import refinitiv.data as rd
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

rd.open_session()

In [ ]:
start_date = datetime(2008, 1, 1)
end_date   = datetime(2025, 12, 31)
SLEEP_BETWEEN = 3

## Load Part 1 Checkpoint

In [ ]:
daily_long = pd.read_csv('idx_daily_prices.csv', parse_dates=['Date'])
fund_clean = pd.read_csv('idx_fundamentals.csv', parse_dates=['Date', 'IPO_Date'])
turnover_df = pd.read_csv('idx_turnover_analysis.csv')
all_rics = pd.read_csv('idx_ric_list.csv')['RIC'].tolist()

print(f'Daily: {len(daily_long):,} rows, {daily_long["Instrument"].nunique()} stocks')
print(f'Fundamentals: {len(fund_clean):,} rows')
print(f'Universe: {len(all_rics)} RICs')

## 4. Download JCI Index Returns

In [ ]:
jci_raw = rd.get_history(
    universe=['.JKSE'],
    fields=['TR.PriceClose'],
    interval='1D',
    start=start_date.strftime('%Y-%m-%d'),
    end=end_date.strftime('%Y-%m-%d')
)

jci = jci_raw.reset_index()
jci.columns = ['Date', 'JCI_Close']
jci['JCI_Close'] = pd.to_numeric(jci['JCI_Close'], errors='coerce')
jci['Market_Return'] = np.log(jci['JCI_Close'] / jci['JCI_Close'].shift(1))
jci = jci.dropna(subset=['Market_Return']).reset_index(drop=True)

print(f'JCI index: {len(jci):,} trading days')
print(f'Range: {jci["Date"].min().date()} to {jci["Date"].max().date()}')

## 5. Bank Indonesia Risk-Free Rate

In [ ]:
rf_raw = rd.get_history(
    universe=['IDCBRR=ECI'],
    fields=['VALUE'],
    interval='monthly',
    start=start_date.strftime('%Y-%m-%d'),
    end=end_date.strftime('%Y-%m-%d')
)

daily_range = pd.date_range(start=start_date, end=end_date, freq='D')
rf_daily = rf_raw.reindex(daily_range).ffill().bfill().reset_index()
rf_daily.columns = ['Date', 'Annual_BI_Rate']
rf_daily['Annual_BI_Rate'] = pd.to_numeric(rf_daily['Annual_BI_Rate'], errors='coerce')
rf_daily['Daily_Rf'] = (rf_daily['Annual_BI_Rate'] / 100) / 360

print(f'Risk-free rate: {len(rf_daily):,} daily obs')
print(f'BI Rate range: {rf_daily["Annual_BI_Rate"].min():.2f}% – {rf_daily["Annual_BI_Rate"].max():.2f}%')

## 6. Assemble Master Panel

In [ ]:
daily_long['Date'] = pd.to_datetime(daily_long['Date'])
jci['Date'] = pd.to_datetime(jci['Date'])
rf_daily['Date'] = pd.to_datetime(rf_daily['Date'])

master = pd.merge(daily_long, jci[['Date', 'Market_Return']], on='Date', how='left')
master = pd.merge(master, rf_daily[['Date', 'Daily_Rf']], on='Date', how='left')

master = master.sort_values(['Instrument', 'Date'])
master['Stock_Return'] = master.groupby('Instrument')['Price'].transform(
    lambda x: np.log(x / x.shift(1))
)
master['Excess_Return'] = master['Stock_Return'] - master['Daily_Rf']
master['Market_Excess'] = master['Market_Return'] - master['Daily_Rf']
master['DayOfWeek'] = master['Date'].dt.dayofweek
master['DayName'] = master['Date'].dt.day_name()
master['YearMonth'] = master['Date'].dt.to_period('M')

print(f'Master panel: {len(master):,} rows, {master["Instrument"].nunique()} stocks')
print(f'Columns: {list(master.columns)}')

## 7. Save Final Datasets

In [ ]:
jci.to_csv('jci_index.csv', index=False)
rf_daily.to_csv('bi_riskfree_rate.csv', index=False)
master.to_csv('idx_master_panel.csv', index=False)

print('Saved:')
print(f'  jci_index.csv          — {len(jci):,} rows')
print(f'  bi_riskfree_rate.csv   — {len(rf_daily):,} rows')
print(f'  idx_master_panel.csv   — {len(master):,} rows')

## 8. Data Quality Report

In [ ]:
print('=' * 60)
print('DATA QUALITY REPORT')
print('=' * 60)

print(f'\n── Survivorship Bias ──')
print(f'Historical universe: {len(all_rics)} RICs')
print(f'Avg annual turnover: {turnover_df["Turnover_Pct"].mean():.1f}%')

print(f'\n── Daily Prices ──')
print(f'Rows: {len(daily_long):,} | Stocks: {daily_long["Instrument"].nunique()}')
print(f'Range: {daily_long["Date"].min().date()} to {daily_long["Date"].max().date()}')
obs = daily_long.groupby('Instrument').size()
print(f'Obs/stock: median={obs.median():.0f}, min={obs.min()}, max={obs.max()}')

print(f'\n── Fundamentals ──')
print(f'Rows: {len(fund_clean):,} | Stocks: {fund_clean["Instrument"].nunique()}')

print(f'\n── Market Data ──')
print(f'JCI trading days: {len(jci):,}')
print(f'BI Rate: {rf_daily["Annual_BI_Rate"].min():.2f}% – {rf_daily["Annual_BI_Rate"].max():.2f}%')

print(f'\n── Master Panel ──')
print(f'Rows: {len(master):,} | Stocks: {master["Instrument"].nunique()}')
print(master[['Price','Volume','Stock_Return','Market_Return','Daily_Rf']].notnull().mean() * 100)